<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='1.%20what_postgresql_is.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Previous</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='3.%20first_steps_with_psql.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Next →</a>
</div>

# Chapter 2: Installing PostgreSQL

Getting a correctly configured PostgreSQL installation is the foundation everything else depends on. This chapter covers the recommended installation methods, directory layout, and the critical configuration files you will touch on day one.

## The Big Picture

Installing PostgreSQL is like setting up a new apartment. There is the building itself (the PostgreSQL binaries), the apartment unit (the data cluster — `PGDATA`), the utilities (configuration files), and the keys (roles and authentication). If you install it carelessly, you will spend months working around misconfiguration.

The most important principle: **use official packages from postgresql.org**, not the outdated versions bundled with operating systems. Ubuntu's `apt-get install postgresql` often installs a version two or three major releases behind. Always use the PGDG (PostgreSQL Global Development Group) repository so you control the version.

A **PostgreSQL cluster** (also called an instance) is a single `postgres` server process managing one data directory, one port, and multiple databases. Every environment—development, staging, production—should run its own cluster. Sharing a cluster between environments creates dependency chains that complicate upgrades and backups.

The data directory (`PGDATA`) contains everything: the actual data pages, WAL segments, configuration files, and access control lists. Knowing its layout prevents panic during incidents and is required knowledge for backup and recovery.

## Core Concepts

### Installation Methods
Three primary methods exist: OS package manager (PGDG repository), official Docker image, and `brew` on macOS. Each has appropriate use cases—PGDG packages for Linux servers, Docker for local development parity, brew for macOS convenience.

### The PGDATA Directory Layout
- `base/` — one subdirectory per database (named by OID); actual table and index files live here
- `global/` — cluster-wide tables (pg_database, pg_roles)
- `pg_wal/` — Write-Ahead Log segments; never delete manually
- `pg_hba.conf` — Host-Based Authentication rules; controls who can connect
- `postgresql.conf` — Server configuration parameters (memory, connections, logging)
- `pg_ident.conf` — OS username to PostgreSQL role mapping

### postgresql.conf Critical Parameters
The defaults are conservative (suitable for a small VM). Production systems require tuning of `shared_buffers`, `work_mem`, `max_connections`, `effective_cache_size`, and `wal_level`.

### pg_hba.conf Authentication Methods
`trust` (no password, dangerous), `md5` (hashed password, legacy), `scram-sha-256` (modern default), `peer` (OS user match, local only), `cert` (TLS client certificate).

### Service Management
On Linux (systemd): `systemctl start/stop/restart postgresql`. The service name includes the version, e.g., `postgresql-16`. On macOS (brew): `brew services start postgresql@16`.

## Key SQL Syntax

```sql
-- Check running PostgreSQL version
SELECT version();
SELECT current_setting('server_version');

-- View important configuration parameters
SHOW max_connections;
SHOW shared_buffers;
SHOW work_mem;
SHOW data_directory;
SHOW log_directory;

-- View all non-default settings
SELECT name, setting, unit, context
FROM pg_settings
WHERE source != 'default'
ORDER BY name;

-- Reload configuration without restart (for most parameters)
SELECT pg_reload_conf();

-- View authentication configuration (pg_hba rules in effect)
SELECT type, database, user_name, address, auth_method
FROM pg_hba_file_rules;
```

In [ ]:
# Installation methods and their appropriate use cases

installation_methods = {
    "PGDG APT (Ubuntu/Debian)": {
        "commands": [
            "sudo apt install -y postgresql-common",
            "sudo /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh",
            "sudo apt-get install -y postgresql-16",
        ],
        "use_case": "Production Linux servers — official packages, up-to-date versions",
        "pros": "Systemd integration, official support, security patches",
        "cons": "Tied to distro package manager",
    },
    "PGDG YUM/DNF (RHEL/CentOS/Fedora)": {
        "commands": [
            "sudo dnf install -y https://download.postgresql.org/pub/repos/yum/reporpms/EL-9-x86_64/pgdg-redhat-repo-latest.noarch.rpm",
            "sudo dnf -qy module disable postgresql",
            "sudo dnf install -y postgresql16-server",
            "sudo /usr/pgsql-16/bin/postgresql-16-setup initdb",
            "sudo systemctl enable --now postgresql-16",
        ],
        "use_case": "RHEL-based production servers",
        "pros": "Enterprise support channels available",
        "cons": "More setup steps than Debian",
    },
    "Docker (official image)": {
        "commands": [
            "docker run -d --name pg16 \\",
            "  -e POSTGRES_PASSWORD=devpass \\",
            "  -e POSTGRES_USER=dev \\",
            "  -e POSTGRES_DB=appdb \\",
            "  -p 5432:5432 \\",
            "  -v pgdata:/var/lib/postgresql/data \\",
            "  postgres:16-alpine",
        ],
        "use_case": "Local development, CI/CD pipelines, integration tests",
        "pros": "Exact version control, reproducible, fast teardown/rebuild",
        "cons": "Volume management needed for data persistence",
    },
    "Homebrew (macOS)": {
        "commands": [
            "brew install postgresql@16",
            "brew services start postgresql@16",
            'echo 'export PATH="/opt/homebrew/opt/postgresql@16/bin:$PATH"' >> ~/.zshrc',
        ],
        "use_case": "macOS local development",
        "pros": "Simple install and update",
        "cons": "macOS only, version lags slightly",
    },
}

for method, details in installation_methods.items():
    print(f"=== {method} ===")
    print(f"  Use case: {details['use_case']}")
    print(f"  Pros: {details['pros']}")
    print(f"  Cons: {details['cons']}")
    print("  Commands:")
    for cmd in details["commands"]:
        print(f"    {cmd}")
    print()

## Deep Dive with Examples

### Example 1: PGDATA Directory Layout

Understanding the `PGDATA` layout is required for backup, recovery, and debugging. These files are the source of truth for everything your PostgreSQL cluster knows.

In [ ]:
# PGDATA directory structure explained

pgdata_layout = {
    "base/": {
        "description": "Per-database subdirectories named by OID. Table and index heap files live here.",
        "example": "base/16384/ → database OID 16384 (your appdb)",
        "critical": True,
    },
    "global/": {
        "description": "Cluster-wide system tables: pg_database, pg_tablespace, pg_authid.",
        "example": "global/pg_control → cluster state file (checked on startup)",
        "critical": True,
    },
    "pg_wal/": {
        "description": "Write-Ahead Log segments (16MB each by default). NEVER delete manually.",
        "example": "pg_wal/000000010000000000000001 → first WAL segment",
        "critical": True,
    },
    "pg_hba.conf": {
        "description": "Host-Based Authentication. Controls who can connect and how.",
        "example": "host all all 0.0.0.0/0 scram-sha-256",
        "critical": True,
    },
    "postgresql.conf": {
        "description": "Server configuration. Most parameters require reload; some need restart.",
        "example": "shared_buffers = 256MB",
        "critical": True,
    },
    "pg_ident.conf": {
        "description": "Maps OS usernames to PostgreSQL role names for ident/peer auth.",
        "example": "mymap   os_admin   pg_superuser",
        "critical": False,
    },
    "pg_log/ (or log/)": {
        "description": "Server log files. Location controlled by log_directory parameter.",
        "example": "postgresql-2024-01-15_000000.log",
        "critical": False,
    },
    "pg_stat_tmp/": {
        "description": "Temporary statistics files for pg_stat_* views. Rebuilt on startup.",
        "example": "global.stat → cumulative statistics",
        "critical": False,
    },
}

print("=== PGDATA Directory Layout ===")
print()
for path, info in pgdata_layout.items():
    flag = "⚠ CRITICAL" if info["critical"] else "  optional"
    print(f"  [{flag}] {path}")
    print(f"           {info['description']}")
    print(f"           Example: {info['example']}")
    print()

## Deep Dive with Examples

### Example 2: postgresql.conf — Essential Parameters

The default configuration is intentionally conservative. For any workload beyond a toy project, these parameters must be tuned. A good starting point is [PGTune](https://pgtune.leopard.in.ua/) — it generates configuration based on hardware specs.

In [ ]:
# Essential postgresql.conf parameters with explanations

parameters = [
    {
        "name": "shared_buffers",
        "default": "128MB",
        "recommendation": "25% of RAM",
        "example": "shared_buffers = 4GB  # for 16GB RAM server",
        "restart_required": True,
        "why": "PostgreSQL's buffer pool — most important memory parameter",
    },
    {
        "name": "work_mem",
        "default": "4MB",
        "recommendation": "RAM / (max_connections * 2)",
        "example": "work_mem = 64MB  # per sort/hash operation",
        "restart_required": False,
        "why": "Per-operation sort/hash memory; set too high with many connections = OOM",
    },
    {
        "name": "max_connections",
        "default": "100",
        "recommendation": "Use PgBouncer; keep this ≤ 200",
        "example": "max_connections = 100  # use connection pooler instead",
        "restart_required": True,
        "why": "Each connection uses ~5MB RAM; high counts waste memory and slow planning",
    },
    {
        "name": "effective_cache_size",
        "default": "4GB",
        "recommendation": "75% of RAM",
        "example": "effective_cache_size = 12GB  # hint for query planner",
        "restart_required": False,
        "why": "Planner hint for OS cache size; doesn't allocate memory, just informs decisions",
    },
    {
        "name": "wal_level",
        "default": "replica",
        "recommendation": "replica (default) or logical for replication",
        "example": "wal_level = replica  # required for streaming replication",
        "restart_required": True,
        "why": "Controls WAL content; logical adds column-level change data",
    },
    {
        "name": "log_min_duration_statement",
        "default": "-1 (disabled)",
        "recommendation": "1000 (log queries over 1 second)",
        "example": "log_min_duration_statement = 1000  # milliseconds",
        "restart_required": False,
        "why": "Essential for production monitoring; surfaces slow queries automatically",
    },
    {
        "name": "timezone",
        "default": "varies by OS",
        "recommendation": "UTC always",
        "example": "timezone = 'UTC'",
        "restart_required": False,
        "why": "Consistent timezone prevents timestamp bugs in applications",
    },
]

print("=== Essential postgresql.conf Parameters ===")
for p in parameters:
    restart = "RESTART required" if p["restart_required"] else "reload with pg_reload_conf()"
    print(f"  {p['name']}")
    print(f"    Default:        {p['default']}")
    print(f"    Recommendation: {p['recommendation']}")
    print(f"    Example:        {p['example']}")
    print(f"    Change:         {restart}")
    print(f"    Why:            {p['why']}")
    print()

## Deep Dive with Examples

### Example 3: pg_hba.conf — Authentication Rules

`pg_hba.conf` is read top-to-bottom; the first matching rule wins. Misconfiguration here is the most common reason new installations fail with "FATAL: password authentication failed" or "FATAL: no pg_hba.conf entry for host".

In [ ]:
# pg_hba.conf authentication methods and rules

hba_format = "TYPE  DATABASE  USER  ADDRESS       METHOD"
print(f"Format: {hba_format}")
print()

hba_examples = [
    {
        "rule": "local   all       all              peer",
        "meaning": "OS users connecting via Unix socket match by OS username (no password)",
        "use_case": "postgres system user running psql on the server itself",
        "security": "High — requires OS-level access",
    },
    {
        "rule": "host    all       all  127.0.0.1/32  scram-sha-256",
        "meaning": "TCP connections from localhost require SCRAM password",
        "use_case": "Local application connecting via TCP (not Unix socket)",
        "security": "High — SCRAM is modern, salted, not replayable",
    },
    {
        "rule": "host    all       all  10.0.0.0/8    scram-sha-256",
        "meaning": "Any host in 10.x.x.x range requires SCRAM password",
        "use_case": "Application servers in same VPC",
        "security": "High — combine with network-level firewall",
    },
    {
        "rule": "hostssl appdb     appuser 0.0.0.0/0  scram-sha-256",
        "meaning": "SSL required; only appuser can connect to appdb from anywhere",
        "use_case": "Production — restricts by database, user, and requires TLS",
        "security": "Highest — TLS + SCRAM + user/db restriction",
    },
    {
        "rule": "host    all       all  0.0.0.0/0     trust",
        "meaning": "Anyone from any IP connects without password",
        "use_case": "NEVER in production — testing only in isolated environments",
        "security": "NONE — do not use",
    },
]

auth_methods = {
    "trust":         "No authentication — never in production",
    "reject":        "Always rejects — useful to block specific users/hosts",
    "md5":           "MD5 hashed password — legacy, avoid for new setups",
    "scram-sha-256": "Modern SASL authentication — use this",
    "peer":          "OS username must match PostgreSQL role — local only",
    "cert":          "TLS client certificate — highest security",
    "ldap":          "Enterprise LDAP/Active Directory integration",
    "radius":        "RADIUS server authentication",
}

print("=== pg_hba.conf Rule Examples ===")
for ex in hba_examples:
    print(f"  Rule:      {ex['rule']}")
    print(f"  Meaning:   {ex['meaning']}")
    print(f"  Use case:  {ex['use_case']}")
    print(f"  Security:  {ex['security']}")
    print()

print("=== Authentication Methods Reference ===")
for method, description in auth_methods.items():
    flag = " ⚠" if method in ("trust", "md5") else ""
    print(f"  {method:<16} {description}{flag}")

## The "What If?" Experimentation Section

1. **What if you set `max_connections = 1000`?** Calculate: 1000 connections × 5MB overhead = 5GB of RAM just for connection overhead, before any queries run. This is why PgBouncer or Pgpool-II (connection poolers) exist.

2. **What if `pg_hba.conf` has no matching rule for your connection?** PostgreSQL returns `FATAL: no pg_hba.conf entry for host "x.x.x.x", user "y", database "z", SSL off`. Try adding a `host all all 127.0.0.1/32 scram-sha-256` rule and reloading with `SELECT pg_reload_conf()`.

3. **What if you never tune `work_mem`?** The default 4MB means complex sorts spill to disk. Try `EXPLAIN (ANALYZE, BUFFERS) SELECT ... ORDER BY large_column` before and after setting `SET work_mem = '256MB'` in your session.

In [ ]:
# Experiment: Calculate memory impact of max_connections settings

def calculate_connection_memory(max_connections, work_mem_mb=4, shared_buffers_mb=256):
    """Estimate PostgreSQL memory usage for different connection counts."""
    per_connection_overhead_mb = 5  # approximate
    connection_overhead = max_connections * per_connection_overhead_mb
    total_baseline_mb = shared_buffers_mb + connection_overhead
    
    return {
        "max_connections": max_connections,
        "connection_overhead_mb": connection_overhead,
        "shared_buffers_mb": shared_buffers_mb,
        "total_baseline_mb": total_baseline_mb,
        "total_baseline_gb": round(total_baseline_mb / 1024, 2),
        "work_mem_worst_case_mb": max_connections * work_mem_mb * 2,  # 2 sorts per connection
    }

configs = [50, 100, 200, 500, 1000]

print("=== Connection Count Memory Impact Analysis ===")
print(f"{'max_conn':>10} {'conn_overhead':>15} {'baseline_GB':>12} {'work_mem_worst':>15}")
print("-" * 55)
for c in configs:
    r = calculate_connection_memory(c)
    warning = " ← use pooler" if c >= 500 else ""
    print(f"{r['max_connections']:>10} {r['connection_overhead_mb']:>13}MB "
          f"{r['total_baseline_gb']:>11}GB "
          f"{r['work_mem_worst_case_mb']:>13}MB{warning}")

print()
print("Recommendation: Keep max_connections ≤ 100-200 and use PgBouncer")
print("for connection pooling. Applications connect to PgBouncer, which")
print("maintains a small pool of actual PostgreSQL connections.")

## Real-World Project Link

**Mini project:** Create a Docker Compose file for a local PostgreSQL development environment.

Requirements:
- Use `postgres:16-alpine` image
- Set `POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB` via environment variables
- Mount a named volume for data persistence
- Add a `healthcheck` using `pg_isready`
- Include a bind mount for an `init.sql` file that creates your schema on first startup

Bonus: Add a second service using `adminer` or `pgAdmin4` for visual inspection. This exercise builds muscle memory for the Docker-based workflow most teams use for local development.

## Chapter Summary & Cheat Sheet

### What you learned
- Always install from PGDG packages — never OS-bundled PostgreSQL
- `PGDATA` contains data, WAL, config, and auth files — understand its layout
- `postgresql.conf` key parameters: `shared_buffers` (25% RAM), `work_mem`, `max_connections`, `timezone = 'UTC'`
- `pg_hba.conf` is read top-down; `scram-sha-256` is the modern auth method
- Docker with named volumes is ideal for local development environments

### Cheat sheet
```bash
# PGDG install (Ubuntu)
sudo /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh
sudo apt-get install -y postgresql-16

# Docker quick start
docker run -d --name pg16 \
  -e POSTGRES_PASSWORD=devpass \
  -p 5432:5432 postgres:16-alpine

# Service management (systemd)
sudo systemctl start postgresql-16
sudo systemctl status postgresql-16

# Connect as postgres user
sudo -u postgres psql

# Reload config after changes
SELECT pg_reload_conf();

# View non-default settings
SELECT name, setting FROM pg_settings WHERE source != 'default';
```

## Practice Prompts

1. What is the difference between a PostgreSQL **cluster** and a **database**? Can you query across databases within the same cluster without extensions?
2. List three `postgresql.conf` parameters that require a server restart versus those that can be applied with `pg_reload_conf()`.
3. A developer reports "FATAL: password authentication failed for user 'app'". Walk through the `pg_hba.conf` debugging process step by step.
4. Why should `work_mem` be set conservatively when `max_connections` is high? Write the formula for worst-case memory consumption.
5. Create a minimal `docker-compose.yml` for a PostgreSQL 16 development environment with a named volume and environment-variable credentials.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='1.%20what_postgresql_is.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Previous</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='3.%20first_steps_with_psql.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Next →</a>
</div>